# rushlite playground
random snippets to poke at the public API — tensors, autograd, and the `nets` layers.

In [1]:
import rushlite
from rushlite._C import Variable
from rushlite.nets.layers import Linear, ReLU, Sigmoid, Tanh, Softmax, Sequential, Dropout, Flatten

## tensor ops + autograd

In [2]:
x = Variable([[1., 2.], [3., 4.]], requires_grad=True)
z = x @ x.T
z.tolist(), z.shape

([5.0, 11.0, 11.0, 25.0], [2, 2])

In [3]:
loss = (x * x).sum(0).sum(0)
loss.backward()
x.grad   # d/dx sum(x^2) = 2x

Tensor(data=[2, 4, 6, 8], shape=[2, 2], dtype=Float32, device=CPU)

## helpers: hand-rolled SGD (no optim module yet)
params are updated in place via `Tensor.copy`.

In [4]:
def sgd_step(params, lr):
    for p in params:
        p.data.copy((p - lr * Variable(p.grad)).data)
        p.zero_grad()

def mean_all(v):
    n = 1
    for d in v.shape: n *= d
    for ax in range(v.ndim): v = v.sum(ax)
    return v / n

def mse(pred, target):
    return mean_all((pred - target) * (pred - target))

## linear regression — should recover w=[2, -3], b=1

In [5]:
xs = [[float(i), float(j)] for i in range(-2, 3) for j in range(-2, 3)]
ys = [[2*a - 3*b + 1] for a, b in xs]
X, Y = Variable(xs), Variable(ys)

model = Linear(2, 1)
for epoch in range(300):
    loss = mse(model(X), Y)
    loss.backward()
    sgd_step(list(model.parameters()), 0.05)

print("loss:", loss.tolist()[0])
print("w:", model.weights.tolist(), " b:", model.bias.tolist())

loss: 7.770495401640543e-13
w: [1.999999761581421, -2.999999523162842]  b: [0.9999997019767761]


## XOR with a lil MLP

In [6]:
X = Variable([[0.,0.], [0.,1.], [1.,0.], [1.,1.]])
Y = Variable([[0.], [1.], [1.], [0.]])

model = Sequential([Linear(2, 8), Tanh(), Linear(8, 1), Sigmoid()])
params = list(model.parameters())
for epoch in range(2000):
    loss = mse(model(X), Y)
    loss.backward()
    sgd_step(params, 0.5)
    if epoch % 400 == 0: print(epoch, loss.tolist()[0])

[round(p, 3) for p in model(X).tolist()]   # want ~[0, 1, 1, 0]

0 0.48235881328582764
400 0.0032715003471821547
800 0.0013346405467018485
1200 0.0008116069948300719
1600 0.0005754356971010566


[0.019, 0.977, 0.979, 0.021]

## 3-class softmax classifier (cross-entropy)

In [7]:
anchors = [(-2., -2.), (2., -2.), (0., 2.)]
pts, hot = [], []
for c, (ax, ay) in enumerate(anchors):
    for dx, dy in [(0, 0), (.3, .1), (-.2, .2), (.1, -.3)]:
        pts.append([ax+dx, ay+dy])
        hot.append([1. if i == c else 0. for i in range(3)])
X, Y = Variable(pts), Variable(hot)

clf = Sequential([Linear(2, 16), Tanh(), Linear(16, 3), Softmax(-1)])
for epoch in range(400):
    p = clf(X)
    loss = -mean_all(Y * p.clamp(1e-7, 1.0).log())
    loss.backward()
    sgd_step(list(clf.parameters()), 0.5)

probs = clf(X).tolist()
rows = [probs[i*3:(i+1)*3] for i in range(len(pts))]
acc = sum(r.index(max(r)) == h.index(1.0) for r, h in zip(rows, hot)) / len(pts)
print("train acc:", acc)

train acc: 1.0


## layer quirks
heads up: `Dropout(p)` — p is the **keep** probability (mirrors the C++), and there's no eval-mode / rescaling.

In [8]:
x = Variable([[1., 2.], [3., 4.]])
Dropout(1.0)(x).tolist(), Dropout(0.5)(x).tolist(), Dropout(0.0)(x).tolist()

([1.0, 2.0, 3.0, 4.0], [1.0, 0.0, 3.0, 4.0], [0.0, 0.0, 0.0, 0.0])

In [9]:
imgs = rushlite.rand([2, 4, 4])   # fake batch of 4x4 "images"
Flatten()(imgs).shape

[2, 16]

## inspecting a model

In [10]:
mlp = Sequential([Flatten(), Linear(16, 64), ReLU(), Linear(64, 10), Softmax(-1)])
for name, p in mlp.named_parameters():
    print(name, p.shape)

1.weights [16, 64]
1.bias [64]
3.weights [64, 10]
3.bias [10]


## capture mode (kernel fusion) vs eager — same numbers?

In [11]:
X = rushlite.rand([64, 32])
mlp = Sequential([Linear(32, 64), ReLU(), Linear(64, 10), Softmax(-1)])

eager = mlp(X).tolist()
with rushlite.capture_on():
    fused = mlp(X).tolist()
max(abs(a - b) for a, b in zip(eager, fused))

0.0